<div style="background:linear-gradient(135deg,#4F46E5 0%,#7C3AED 55%,#9333EA 100%);padding:30px 34px;border-radius:16px;color:white;margin-bottom:14px"><div style="font-size:12px;letter-spacing:3px;text-transform:uppercase;opacity:.85">INF672 · Machine Learning · Probabilidad</div><h1 style="margin:10px 0 6px;font-size:34px;line-height:1.1">GL06 · Dos formas de medir el azar</h1><p style="margin:0;font-size:15px;opacity:.92">El enfoque clásico (contar) y el frecuentista (repetir) — con simulaciones en Python</p></div><div style="background:#F8FAFC;border:1px solid #E2E8F0;border-radius:12px;padding:14px 20px;display:flex;flex-wrap:wrap;gap:8px 30px;font-size:14px"><span><b>Estudiante:</b> Abdair Magdiel Coca Carlo</span><span><b>Materia:</b> INF672 · Ing. Informática · UATF</span><span><b>Docente:</b> M.Sc. Huáscar Fedor Gonzales Guzmán</span><span><b>Unidad:</b> Fundamentos matemáticos del aprendizaje</span><span><b>Gestión:</b> 2026</span></div>

<h3 style="margin-top:0">Contenido del cuaderno</h3><div style="margin:7px 0;font-size:14.5px"><span style="display:inline-block;background:#4F46E5;color:white;border-radius:50%;width:26px;height:26px;text-align:center;line-height:26px;font-weight:700;font-size:13px;margin-right:9px">1</span><b>Entorno</b> — NumPy, Matplotlib, itertools, Counter, paleta INF672.</div><div style="margin:7px 0;font-size:14.5px"><span style="display:inline-block;background:#7C3AED;color:white;border-radius:50%;width:26px;height:26px;text-align:center;line-height:26px;font-weight:700;font-size:13px;margin-right:9px">2</span><b>Marco conceptual</b> — clásica (Laplace, a priori) vs frecuentista (a posteriori) vs bayesiana.</div><div style="margin:7px 0;font-size:14.5px"><span style="display:inline-block;background:#F59E0B;color:white;border-radius:50%;width:26px;height:26px;text-align:center;line-height:26px;font-weight:700;font-size:13px;margin-right:9px">3</span><b>Parte 1 · Clásico</b> — espacio muestral 36 pares, regla Laplace, distribución triangular, mapa calor 6×6, verificación Monte Carlo 50k.</div><div style="margin:7px 0;font-size:14.5px"><span style="display:inline-block;background:#10B981;color:white;border-radius:50%;width:26px;height:26px;text-align:center;line-height:26px;font-weight:700;font-size:13px;margin-right:9px">4</span><b>Parte 2 · Frecuentista</b> — frecuencia acumulada, moneda (Acto 1), chincheta P=0.62 (Acto 2), fragilidad vs N, varianza de la estimación (6 corridas).</div><div style="margin:7px 0;font-size:14.5px"><span style="display:inline-block;background:#EC4899;color:white;border-radius:50%;width:26px;height:26px;text-align:center;line-height:26px;font-weight:700;font-size:13px;margin-right:9px">5</span><b>Síntesis y evaluación</b> — tabla comparativa y rúbrica.</div><div style="background:#EFF6FF;border-left:5px solid #3B82F6;border-radius:8px;padding:12px 16px;margin:14px 0;font-size:14px;color:#334155">Esta guía reconstruye paso a paso el cuaderno de la práctica. Cada bloque va con explicación conceptual ampliada + desglose línea por línea. Ejecutar en orden con <code>Shift+Enter</code> y contrastar salidas.</div>

<div style="background:#F8FAFC;border:1px solid #E2E8F0;border-radius:10px;padding:14px 18px"><div style="font-weight:800;color:#1E293B;margin-bottom:6px">01 · Objetivos de aprendizaje</div><ul style="margin:0 0 0 18px;font-size:14px;color:#334155;line-height:1.7"><li>Distinguir interpretación <b>clásica</b> (a priori, conteo) y <b>frecuentista</b> (a posteriori, repetición).</li><li>Aplicar <b>regla de Laplace</b> enumerando espacio muestral equiprobable y casos favorables.</li><li>Reconocer <b>condición de simetría</b> y por qué su ausencia invalida el clásico.</li><li>Implementar <b>simulaciones Monte Carlo</b> con NumPy para estimar probabilidades.</li><li>Visualizar <b>convergencia</b> (ley grandes números) y varianza de la estimación.</li><li>Conectar con ML: prior uniforme, máxima verosimilitud, valor de los datos, anticipo del sobreajuste.</li></ul></div>

<div style="background:linear-gradient(90deg,#4F46E5 0%,#6366F1 100%);color:white;padding:12px 20px;border-radius:10px;font-size:20px;font-weight:800;margin:16px 0 8px;display:flex;align-items:center;gap:10px"><span style="background:rgba(255,255,255,.22);border-radius:8px;padding:2px 10px;font-size:14px;font-weight:700">02</span>Entorno y preparación</div><p style="font-size:14px;color:#334155">Cuatro herramientas: <b>NumPy</b> (arreglos y azar vectorizado), <b>Matplotlib</b> (gráficos), <code>itertools.product</code> (combinaciones) y <code>collections.Counter</code> (frecuencias). Fijamos identidad visual INF672.</p>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from itertools import product
from collections import Counter

# --- Identidad visual INF672 "cuaderno científico" ---
INDIGO = "#4F46E5"  # primario
PURPURA = "#7E22CE"  # apoyo
CORAL = "#F43F5E"  # errores / advertencias / énfasis
TINTA = "#111827"  # texto
CELDA = "#F9FAFB"  # fondo de celda
HAIRLINE = "#E5E7EB"
mpl.rcParams.update({
    "font.family": ["DejaVu Sans", "sans-serif"],
    "font.size": 11,
    "axes.facecolor": CELDA,
    "figure.facecolor": "white",
    "axes.grid": True,
    "grid.color": HAIRLINE,
})
# Semilla: hace el azar reproducible (mismo resultado en cada corrida).
rng = np.random.default_rng(672)
print("Entorno listo. Paleta INF672 cargada.")

<div style="background:#F8FAFC;border:1px solid #E2E8F0;border-radius:10px;padding:14px 18px;font-size:14px;color:#334155"><b>Cómo funciona:</b><ul style="margin:8px 0 0 18px"><li>Alias cortos <code>np, plt</code> — convención universal.</li><li>Constantes hex centralizan paleta; <code>mpl.rcParams.update</code> fija estilo global (tipografía, fondo ejes, rejilla) para todas las figuras.</li><li><code>np.random.default_rng(672)</code> — generador moderno con semilla fija 672; secuencia idéntica cada corrida para reproducir guía. Preferible a <code>np.random.seed</code>.</li></ul></div>

<div style="background:linear-gradient(90deg,#7C3AED 0%,#9333EA 100%);color:white;padding:12px 20px;border-radius:10px;font-size:20px;font-weight:800;margin:16px 0 8px;display:flex;align-items:center;gap:10px"><span style="background:rgba(255,255,255,.22);border-radius:8px;padding:2px 10px;font-size:14px;font-weight:700">03</span>Marco conceptual: ¿qué significa “probabilidad”?</div><div style="display:grid;grid-template-columns:1fr 1fr;gap:12px"><div style="border-left:5px solid #4F46E5;background:#EFF6FF;border-radius:8px;padding:12px 14px"><div style="color:#4F46E5;font-weight:800;font-size:12px;letter-spacing:1px">INTERPRETACIÓN CLÁSICA (A PRIORI)</div><div style="font-size:13.5px;color:#334155;margin-top:4px">Laplace S. XVIII. Probabilidad como <b>estructura</b>: si hay N resultados igualmente posibles por simetría, cada uno 1/N. Se calcula <i>antes</i> de experimentar. Exacta cuando hay simetría.</div></div><div style="border-left:5px solid #7E22CE;background:#F5F3FF;border-radius:8px;padding:12px 14px"><div style="color:#7E22CE;font-weight:800;font-size:12px;letter-spacing:1px">INTERPRETACIÓN FRECUENTISTA (A POSTERIORI)</div><div style="font-size:13.5px;color:#334155;margin-top:4px">S. XIX–XX. Probabilidad como <b>repetición</b>: fracción de veces que ocurre el evento en muchísimas repeticiones. Se mide <i>durante</i> el experimento. Siempre estimación, nunca valor perfecto.</div></div></div><p style="font-size:14px;color:#334155;margin:10px 0">Existe una tercera vía, la <b>bayesiana/subjetiva</b> (grado de creencia que se actualiza con evidencia), que se retoma más adelante.</p><table style="width:100%;border-collapse:collapse;font-size:14px;margin:8px 0"><thead><tr style="background:#4F46E5;color:white"><th style="padding:8px 12px;text-align:left"></th><th style="padding:8px 12px;text-align:left">Enfoque clásico</th><th style="padding:8px 12px;text-align:left">Enfoque frecuentista</th></tr></thead><tbody><tr style="border-bottom:1px solid #E5E7EB"><td style="padding:8px 12px"><b>El gesto</b></td><td style="padding:8px 12px">Contar</td><td style="padding:8px 12px">Repetir y observar</td></tr><tr style="background:#F9FAFB;border-bottom:1px solid #E5E7EB"><td style="padding:8px 12px"><b>¿Cuándo?</b></td><td style="padding:8px 12px">Antes de experimentar</td><td style="padding:8px 12px">Durante el experimento</td></tr><tr style="border-bottom:1px solid #E5E7EB"><td style="padding:8px 12px"><b>Necesita</b></td><td style="padding:8px 12px">Simetría (equiprobabilidad)</td><td style="padding:8px 12px">Muchísimas repeticiones</td></tr><tr style="background:#F9FAFB;border-bottom:1px solid #E5E7EB"><td style="padding:8px 12px"><b>Entrega</b></td><td style="padding:8px 12px">Valor exacto</td><td style="padding:8px 12px">Estimación</td></tr><tr style="border-bottom:1px solid #E5E7EB"><td style="padding:8px 12px"><b>Se rompe si</b></td><td style="padding:8px 12px">No hay simetría</td><td style="padding:8px 12px">Hay pocos datos</td></tr><tr><td style="padding:8px 12px"><b>Experimento</b></td><td style="padding:8px 12px">Suma de dos dados</td><td style="padding:8px 12px">Moneda y chincheta</td></tr></tbody></table><div style="border-left:5px solid #F59E0B;background:#FFFBEB;border-radius:8px;padding:12px 16px;margin:10px 0"><span style="color:#D97706;font-weight:800;font-size:12px;letter-spacing:2px">IDEA CENTRAL</span><div style="font-size:14px;color:#334155">El clásico es exacto pero <b>frágil</b> (se rompe sin simetría); ahí toma el relevo el frecuentista — donde vive el ML.</div></div>

<div style="background:linear-gradient(90deg,#F59E0B 0%,#F97316 100%);color:white;padding:12px 20px;border-radius:10px;font-size:20px;font-weight:800;margin:16px 0 8px;display:flex;align-items:center;gap:10px"><span style="background:rgba(255,255,255,.22);border-radius:8px;padding:2px 10px;font-size:14px;font-weight:700">04</span>Parte 1 · El enfoque clásico: contar sin tirar</div><p style="font-size:14px;color:#334155">Regla de <b>Laplace</b>: <code>P(A) = favorables / posibles</code> — pero los posibles deben ser <b>igualmente posibles</b>.</p><div style="border-left:5px solid #EF4444;background:#FEF2F2;border-radius:8px;padding:12px 16px;margin:10px 0"><span style="color:#EF4444;font-weight:800;font-size:12px;letter-spacing:2px">LA PREGUNTA Y LA TRAMPA</span><div style="font-weight:700;margin:2px 0">¿P(suma de dos dados = 7)?</div><div style="font-size:14px;color:#334155"><b>Respuesta ingenua (mal):</b> sumas 2–12 → 11 valores → P=1/11≈0.091. Error: las sumas <i>no</i> son equiprobables. Lo equiprobable son los 36 pares (d1,d2).</div></div>

<h3 style="color:#7C3AED;margin:14px 0 4px;font-size:17px">Paso 1 — Enumerar el espacio muestral</h3><p style="font-size:14px;color:#334155;margin:6px 0">Cada par (d1,d2) vale 1/36. Generamos los 36 pares como rejilla 6×6.</p>

In [ ]:
# Todos los pares posibles (d1, d2) con cada dado de 1 a 6.
espacio_muestral = list(product(range(1, 7), repeat=2))  # 36 pares
print(f"Total de resultados equiprobables: {len(espacio_muestral)}\n")
# Los mostramos como una rejilla 6x6 para verlos todos de un vistazo.
for d1 in range(1, 7):
    fila = " ".join(f"({d1},{d2})" for d2 in range(1, 7))
    print(fila)

<div style="background:#F8FAFC;border:1px solid #E2E8F0;border-radius:10px;padding:14px 18px;font-size:14px;color:#334155"><ul style="margin:0 0 0 18px"><li><code>product(range(1,7), repeat=2)</code> — producto cartesiano {1..6}×{1..6}. <code>range(1,7)</code> excluye 7.</li><li><code>len</code> confirma 36; f-string incrusta variable; <code>"\n"</code> salto línea.</li><li>Comprensión <code>f"({d1},{d2})" for d2 in range(1,7)</code> + <code>join</code> arma rejilla 6×6.</li></ul></div><div style="border-left:5px solid #111827;background:#F9FAFB;border-radius:8px;padding:12px 16px;margin:10px 0"><span style="color:#111827;font-weight:800;font-size:12px;letter-spacing:2px">IDEA CLAVE</span><div style="font-size:14px;color:#334155">Las 36 casillas tienen prob 1/36. Ahí vive la equiprobabilidad, no en las sumas. Identificar bien el espacio muestral <i>es</i> el enfoque clásico.</div></div>

<h3 style="color:#7C3AED;margin:14px 0 4px;font-size:17px">Paso 2 — Contar los casos favorables</h3><p style="font-size:14px;color:#334155;margin:6px 0">Para cada suma 2–12 contamos pares que la producen y dividimos entre 36. Suma 7: 6 formas (1,6)(2,5)(3,4)(4,3)(5,2)(6,1); suma 2: solo (1,1).</p>

In [ ]:
# Contamos cuántos pares dan cada suma.
conteo = Counter(d1 + d2 for d1, d2 in espacio_muestral)
print(f"{'Suma':>5} {'Pares favorables':>17} {'Probabilidad':>14}")
print("-" * 40)
for suma in range(2, 13):
    favorables = conteo[suma]
    prob = favorables / 36
    print(f"{suma:>5} {favorables:>17} {prob:>10.4f} ({favorables}/36)")
print(f"\nP(suma = 7) = {conteo[7]}/36 = {conteo[7]/36:.4f}")

<div style="background:#F8FAFC;border:1px solid #E2E8F0;border-radius:10px;padding:14px 18px;font-size:14px;color:#334155"><ul style="margin:0 0 0 18px"><li><code>Counter(d1+d2 for d1,d2 in espacio_muestral)</code> — desempaqueta cada tupla, suma y cuenta. <code>conteo[7]==6</code>.</li><li><code>>5 / .4f</code> formatean columnas; <code>favorables/36</code> es Laplace.</li></ul>P real 7 = 6/36≈16.7 %, no 9.1 %. Es la suma más probable porque tiene más pares.</div>

<h3 style="color:#7C3AED;margin:14px 0 4px;font-size:17px">Paso 3 — La forma de la distribución</h3><p style="font-size:14px;color:#334155;margin:6px 0">11 probabilidades forman silueta triangular con pico en 7: sumas centrales tienen más formas que extremos.</p>

In [ ]:
sumas = list(range(2, 13))
probs = [conteo[s] / 36 for s in sumas]
fig, ax = plt.subplots(figsize=(9, 4.5))
barras = ax.bar(sumas, probs, color=INDIGO, edgecolor="white", width=0.75)
barras[sumas.index(7)].set_color(PURPURA)  # resalta el máximo
for s, p in zip(sumas, probs):
    ax.text(s, p + 0.003, f"{p:.3f}", ha="center", va="bottom", fontsize=9)
ax.set_title("Distribución de la suma de dos dados (por conteo)")
ax.set_xlabel("Suma de los dos dados"); ax.set_ylabel("Probabilidad")
ax.set_xticks(sumas)
plt.tight_layout(); plt.show()

<div style="background:#F8FAFC;border:1px solid #E2E8F0;border-radius:10px;padding:14px 18px;font-size:14px;color:#334155"><ul style="margin:0 0 0 18px"><li>Dos listas paralelas <code>sumas / probs</code> vía comprensión.</li><li><code>ax.bar</code> devuelve barras; <code>sumas.index(7)==5</code> pinta púrpura el máximo.</li><li><code>zip</code> + <code>ax.text</code> etiqueta cada barra; <code>tight_layout</code> evita recortes.</li></ul></div>

<h3 style="color:#7C3AED;margin:14px 0 4px;font-size:17px">Paso 4 — Ver por qué, con un mapa de calor</h3><p style="font-size:14px;color:#334155;margin:6px 0">Barras muestran <i>qué</i> pasa; mapa muestra <i>por qué</i>. Cada celda es un par (d1,d2) coloreado por suma. Diagonales agrupan misma suma — 7 es la más larga (6 celdas), 2 y 12 una sola esquina.</p>

In [ ]:
# Matriz 6x6 donde cada celda es la suma d1 + d2.
matriz = np.array([[d1 + d2 for d2 in range(1, 7)] for d1 in range(1, 7)])
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(matriz, cmap="BuPu", origin="upper")
for i in range(6):
    for j in range(6):
        es_siete = (matriz[i, j] == 7)
        ax.text(j, i, matriz[i, j], ha="center", va="center",
                color=CORAL if es_siete else "black", fontweight="bold")
ax.set_xticklabels(range(1, 7)); ax.set_yticklabels(range(1, 7))
ax.set_xlabel("Dado 2"); ax.set_ylabel("Dado 1")
plt.colorbar(im, ax=ax, label="Suma"); plt.show()

<div style="background:#F8FAFC;border:1px solid #E2E8F0;border-radius:10px;padding:14px 18px;font-size:14px;color:#334155"><ul style="margin:0 0 0 18px"><li>Comprensión anidada → <code>np.array</code> → matriz indexable <code>matriz[i,j]</code>.</li><li><code>imshow(cmap="BuPu")</code> mapea valor→color (sumas altas más oscuras).</li><li>Doble for escribe número; <code>CORAL if es_siete else "black"</code> resalta 7.</li></ul>El 7 tiene diagonal más larga: contar bien el espacio equiprobable evita el “1/11”.</div>

<h3 style="color:#7C3AED;margin:14px 0 4px;font-size:17px">Paso 5 — Verificación: ¿y si tiramos de verdad?</h3><p style="font-size:14px;color:#334155;margin:6px 0">Todo fue a priori. Simulamos <b>50 000</b> tiradas y superponemos frecuencias observadas sobre probabilidades por conteo — el conteo ya había <i>predicho</i>; la simulación solo confirma.</p>

In [ ]:
N = 50_000  # número de tiradas simuladas
d1 = rng.integers(1, 7, size=N)  # N dados: enteros de 1 a 6
d2 = rng.integers(1, 7, size=N)
sumas_simuladas = d1 + d2  # suma elemento a elemento
conteo_sim = Counter(sumas_simuladas)
frec_obs = [conteo_sim[s] / N for s in sumas]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(sumas, probs, color=INDIGO, alpha=0.35, label="Clásico (por conteo)")
ax.plot(sumas, frec_obs, "o-", color=CORAL, label=f"Simulado ({N:,} tiradas)")
ax.legend(frameon=False); plt.show()

<div style="background:#F8FAFC;border:1px solid #E2E8F0;border-radius:10px;padding:14px 18px;font-size:14px;color:#334155"><ul style="margin:0 0 0 18px"><li><code>rng.integers(1,7,size=N)</code> — N enteros vectorizados (rápido, sin bucle).</li><li><code>d1+d2</code> suma elemento a elemento; Counter + comprensión → frecuencia.</li><li><code>bar(alpha=0.35)</code> teoría semitransparente; <code>plot("o-")</code> simulación. Coincidencia visual = verificación.</li></ul></div><div style="border-left:5px solid #4F46E5;background:#EFF6FF;border-radius:8px;padding:12px 16px;margin:10px 0"><span style="color:#4F46E5;font-weight:800;font-size:12px;letter-spacing:2px">CONEXIÓN ML</span><div style="font-size:14px;color:#334155">Enumerar espacio y contar es gesto fundamental. Equiprobabilidad = prior uniforme (bayesiano/MAP). Distribución con forma (triangular) emergiendo de piezas uniformes prefigura modelos generativos.</div></div><div style="border-left:5px solid #F59E0B;background:#FFFBEB;border-radius:8px;padding:12px 16px;margin:10px 0"><span style="color:#D97706;font-weight:800;font-size:12px;letter-spacing:2px">LA PARED DEL CLÁSICO</span><div style="font-size:14px;color:#334155">Necesitó saber que 36 pares eran igualmente posibles. ¿Y una <b>chincheta</b> (↑ vs ⤶)? No hay simetría, no hay espacio parejo que enumerar — el clásico queda <b>mudo</b>. Caso normal en mundo real, no excepción. Necesitamos el enfoque que repite y observa: Parte 2.</div></div>

<div style="background:linear-gradient(90deg,#10B981 0%,#06B6D4 100%);color:white;padding:12px 20px;border-radius:10px;font-size:20px;font-weight:800;margin:16px 0 8px;display:flex;align-items:center;gap:10px"><span style="background:rgba(255,255,255,.22);border-radius:8px;padding:2px 10px;font-size:14px;font-weight:700">05</span>Parte 2 · El enfoque frecuentista: repetir y observar</div><p style="font-size:14px;color:#334155">Definición: <code>P(A)=lím(n→∞) veces que ocurrió A / n</code>. Tira muchas veces, cuenta fracción — esa es tu probabilidad. Trampa honesta: nunca llegas a infinito, siempre <b>estimación</b>, nunca valor perfecto. Esa limitación es exactamente donde vive el ML. Dos actos: moneda (terreno conocido) y chincheta (donde es la única salida).</p>

<h3 style="color:#10B981;margin:14px 0 4px;font-size:17px">El gesto frecuentista, en una función</h3><p style="font-size:14px;color:#334155;margin:6px 0">Repetir y devolver frecuencia relativa <b>acumulada</b> tirada por tirada.</p>

In [ ]:
def frecuencia_acumulada(resultados):
    '''Dada una secuencia de 0s y 1s (1 = ocurrió el evento),
    devuelve la fracción acumulada de 1s tras cada tirada.
    Ej.: [1,0,1,1] -> [1.0, 0.5, 0.667, 0.75]
    '''
    resultados = np.asarray(resultados)
    aciertos_acumulados = np.cumsum(resultados)  # 1,1,2,3...
    tiradas = np.arange(1, len(resultados) + 1)  # 1,2,3,4...
    return aciertos_acumulados / tiradas
print(frecuencia_acumulada([1, 0, 1, 1]))

<div style="background:#F8FAFC;border:1px solid #E2E8F0;border-radius:10px;padding:14px 18px;font-size:14px;color:#334155"><ul style="margin:0 0 0 18px"><li><code>np.asarray</code> → arreglo para vectorizar.</li><li><code>np.cumsum</code> → suma acumulada [1,1,2,3] = aciertos.</li><li><code>np.arange(1,len+1)</code> → 1..N = tiradas.</li><li>División elemento a elemento = media acumulada de 0/1 = frecuencia relativa.</li></ul></div>

<h3 style="color:#7C3AED;margin:14px 0 4px;font-size:17px">Acto 1 — La moneda: ganar confianza</h3><p style="font-size:14px;color:#334155;margin:6px 0">Moneda justa ya sabemos P(cara)=0.5 por clásico. Si frecuentista es bueno, debe converger solo a 0.5. 10 000 lanzamientos: al inicio salta, luego se calma.</p>

In [ ]:
N = 10_000
lanzamientos = rng.integers(0, 2, size=N)  # 1 = cara, 0 = cruz
frec = frecuencia_acumulada(lanzamientos)
fig, ax = plt.subplots(figsize=(9.5, 4.5))
ax.plot(np.arange(1, N + 1), frec, color=INDIGO)
ax.axhline(0.5, color=CORAL, linestyle="--", label="Valor real: 0.5")
ax.set_xscale("log")  # escala log en X: ver el caos inicial y la calma final
ax.set_ylim(0, 1); ax.legend(frameon=False); plt.show()
print(f"Estimación final: {frec[-1]:.4f} (el valor real es 0.5).")

<div style="background:#F8FAFC;border:1px solid #E2E8F0;border-radius:10px;padding:14px 18px;font-size:14px;color:#334155"><ul style="margin:0 0 0 18px"><li><code>rng.integers(0,2,size=N)</code> → {0,1} con prob 0.5.</li><li><code>axhline(0.5)</code> referencia visual.</li><li><code>set_xscale("log")</code> estira primeras decenas; sin log el caos inicial quedaría comprimido.</li><li><code>frec[-1]</code> última estimación tras 10 000 tiradas.</li></ul>Lo importante: no descubrir 0.5 sino <b>comprobar que el procedimiento funciona</b> para confiar donde el clásico no llega.</div>

<h3 style="color:#7C3AED;margin:14px 0 4px;font-size:17px">Acto 2 — La chincheta: la necesidad</h3><p style="font-size:14px;color:#334155;margin:6px 0">Aquí <b>no hay fórmula posible</b>. Naturaleza tiene prob verdadera <i>secreta</i>; experimento solo ve 0/1; estimamos por frecuencia y recién al final revelamos el real.</p>

In [ ]:
# --- La naturaleza (secreto): NO mirar este número al razonar ---
P_VERDADERA = 0.62  # probabilidad real de "punta arriba"
N = 10_000
chinchetas = (rng.random(N) < P_VERDADERA).astype(int)  # 0s y 1s
frec = frecuencia_acumulada(chinchetas)
estimacion = frec[-1]
print(f"Tras {N:,} lanzamientos, estimamos P(punta arriba) ~ {estimacion:.4f}")

<div style="background:#F8FAFC;border:1px solid #E2E8F0;border-radius:10px;padding:14px 18px;font-size:14px;color:#334155"><b>El truco de Bernoulli:</b> <code>rng.random(N) < 0.62</code> → booleanos True con prob 0.62 (62% de uniformes caen debajo de 0.62). <code>.astype(int)</code> → 1/0 vectorizado. Código nunca usa <code>P_VERDADERA</code> para estimar, solo naturaleza para generar datos.</div>

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.5))
ax.plot(np.arange(1, N + 1), frec, color=PURPURA, label="Nuestra estimación")
ax.axhline(P_VERDADERA, color=CORAL, linestyle="--",
         label=f"Valor REAL (revelado): {P_VERDADERA}")
ax.set_xscale("log"); ax.set_ylim(0, 1); ax.legend(frameon=False); plt.show()
print(f"Estimación: {estimacion:.4f} | Real: {P_VERDADERA} | Error: {abs(estimacion - P_VERDADERA):.4f}")

<div style="background:#F8FAFC;border:1px solid #E2E8F0;border-radius:10px;padding:14px 18px;font-size:14px;color:#334155">Estimación púrpura converge a 0.62 coral — valor que ninguna fórmula podía dar. Ese 0.62 salió de los <b>datos</b>. Así funciona ML: distribución verdadera invisible, solo muestras, estimamos desde ellas.</div>

<h3 style="color:#F43F5E;margin:14px 0 4px;font-size:17px">El experimento de la fragilidad — ¿cuántos datos hacen falta?</h3><p style="font-size:14px;color:#334155;margin:6px 0">Debilidad crítica: con pocas repeticiones, estimación poco confiable. Comparamos tamaños de muestra.</p>

In [ ]:
tamaños = [10, 100, 1_000, 10_000, 100_000]
estimaciones = []
for n in tamaños:
    muestra = (rng.random(n) < P_VERDADERA).astype(int)
    est = muestra.mean()  # media = frecuencia de 1s
    estimaciones.append(est)
    print(f"{n:>8,} estimación={est:.4f} error={abs(est - P_VERDADERA):.4f}")

<div style="background:#F8FAFC;border:1px solid #E2E8F0;border-radius:10px;padding:14px 18px;font-size:14px;color:#334155">Media de 0/1 = frecuencia. Error ∝ 1/√N: ×100 datos → ÷10 error.</div>

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
# coral si el error es grande (>0.03), índigo si es aceptable
colores = [CORAL if abs(e - P_VERDADERA) > 0.03 else INDIGO for e in estimaciones]
ax.bar([str(n) for n in tamaños], estimaciones, color=colores)
ax.axhline(P_VERDADERA, color=PURPURA, linestyle="--", label=f"Valor real: {P_VERDADERA}")
ax.set_ylim(0, 1); ax.legend(frameon=False); plt.show()

<div style="background:#F8FAFC;border:1px solid #E2E8F0;border-radius:10px;padding:14px 18px;font-size:14px;color:#334155">Comprensión con condicional pinta coral lo poco confiable. <code>str(n)</code> como etiquetas categóricas evita aplastar valores pequeños en escala numérica. Con N=10 estimación pésima; con N=100k sólida.</div>

<h3 style="color:#7C3AED;margin:14px 0 4px;font-size:17px">Un detalle sutil: la estimación misma es azarosa</h3><p style="font-size:14px;color:#334155;margin:6px 0">Repetimos simulación 6 veces con azares distintos: caminos distintos, mismo destino 0.62.</p>

In [ ]:
N = 10_000
n_corridas = 6
fig, ax = plt.subplots(figsize=(9.5, 4.5))
paleta = plt.cm.plasma(np.linspace(0.1, 0.8, n_corridas))
for k in range(n_corridas):
    rng_k = np.random.default_rng(k)  # cada corrida, su propia semilla
    muestra = (rng_k.random(N) < P_VERDADERA).astype(int)
    ax.plot(np.arange(1, N + 1), frecuencia_acumulada(muestra),
            color=paleta[k], alpha=0.85)
ax.axhline(P_VERDADERA, color=CORAL, linestyle="--", label=f"Valor real: {P_VERDADERA}")
ax.set_xscale("log"); ax.set_ylim(0, 1); ax.legend(frameon=False); plt.show()

<div style="background:#F8FAFC;border:1px solid #E2E8F0;border-radius:10px;padding:14px 18px;font-size:14px;color:#334155"><ul style="margin:0 0 0 18px"><li><code>default_rng(k)</code> cada corrida semilla distinta → 6 mundos paralelos.</li><li><code>plt.cm.plasma</code> colores equiespaciados.</li><li>Todas terminan pegadas a 0.62, trayectorias iniciales difieren: límite estable, camino azaroso. Estimar desde muestra finita ⇒ <b>varianza</b>.</li></ul></div><div style="border-left:5px solid #4F46E5;background:#EFF6FF;border-radius:8px;padding:12px 16px;margin:10px 0"><span style="color:#4F46E5;font-weight:800;font-size:12px;letter-spacing:2px">CONEXIÓN ML</span><div style="font-size:14px;color:#334155"><b>Entrenar modelo es esto:</b> estimar probabilidades/parámetros desde frecuencias en datos = máxima verosimilitud.<br><b>Chincheta = mundo real:</b> nunca conoces probs verdaderas, solo muestra.<br><b>“Más datos ayuda”</b> justificado visualmente por convergencia.<br><b>Fragilidad → sobreajuste:</b> estimar con N=10 es peligroso, alta varianza — problema que perseguirá todo el curso.</div></div>

<div style="background:linear-gradient(90deg,#4F46E5 0%,#6366F1 100%);color:white;padding:12px 20px;border-radius:10px;font-size:20px;font-weight:800;margin:16px 0 8px;display:flex;align-items:center;gap:10px"><span style="background:rgba(255,255,255,.22);border-radius:8px;padding:2px 10px;font-size:14px;font-weight:700">06</span>Síntesis: dos gestos para medir el azar</div><table style="width:100%;border-collapse:collapse;font-size:14px"><thead><tr style="background:#4F46E5;color:white"><th style="padding:8px 12px;text-align:left"></th><th style="padding:8px 12px;text-align:left">Clásico (Parte 1)</th><th style="padding:8px 12px;text-align:left">Frecuentista (Parte 2)</th></tr></thead><tbody><tr style="border-bottom:1px solid #E5E7EB"><td style="padding:8px 12px"><b>Cómo obtiene P</b></td><td style="padding:8px 12px">Cuenta favorables / totales</td><td style="padding:8px 12px">Repite y observa frecuencia</td></tr><tr style="background:#F9FAFB;border-bottom:1px solid #E5E7EB"><td style="padding:8px 12px"><b>Cuándo</b></td><td style="padding:8px 12px">A priori, antes de tirar</td><td style="padding:8px 12px">A posteriori, tirando</td></tr><tr style="border-bottom:1px solid #E5E7EB"><td style="padding:8px 12px"><b>Qué necesita</b></td><td style="padding:8px 12px">Simetría (equiprobabilidad)</td><td style="padding:8px 12px">Muchas repeticiones (datos)</td></tr><tr style="background:#F9FAFB;border-bottom:1px solid #E5E7EB"><td style="padding:8px 12px"><b>Qué entrega</b></td><td style="padding:8px 12px">Valor exacto</td><td style="padding:8px 12px">Estimación (nunca perfecta)</td></tr><tr style="border-bottom:1px solid #E5E7EB"><td style="padding:8px 12px"><b>Se rompe cuando</b></td><td style="padding:8px 12px">No hay simetría (chincheta)</td><td style="padding:8px 12px">Hay pocos datos (fragilidad)</td></tr><tr><td style="padding:8px 12px"><b>Ejemplo</b></td><td style="padding:8px 12px">Suma de dos dados</td><td style="padding:8px 12px">Moneda y chincheta</td></tr></tbody></table><div style="background:#FFFBEB;border:1px solid #FDE68A;border-radius:10px;padding:14px 18px;margin-top:12px;font-size:14px;color:#334155">No compiten, se <b>complementan</b>. Clásico ideal exacto con simetría; frecuentista herramienta práctica sin ella. En ML casi nunca hay simetría conocida → vivimos en mundo frecuentista: estimando desde datos con incertidumbre.</div><div style="background:linear-gradient(90deg,#7C3AED 0%,#9333EA 100%);color:white;padding:12px 20px;border-radius:10px;font-size:18px;font-weight:800;margin:16px 0 8px;display:flex;align-items:center;gap:10px">07 · Criterios de evaluación</div><table style="width:100%;border-collapse:collapse;font-size:14px"><thead><tr style="background:#4F46E5;color:white"><th style="padding:8px 12px;text-align:left">Criterio</th><th style="padding:8px 12px;text-align:left">Descripción</th><th style="padding:8px 12px;text-align:left">Peso</th></tr></thead><tbody><tr><td style="padding:8px 12px"><b>Ejecución</b></td><td style="padding:8px 12px">Cuaderno corre completo sin errores y reproduce figuras</td><td style="padding:8px 12px">80%</td></tr><tr style="background:#F9FAFB"><td style="padding:8px 12px"><b>Interpretación</b></td><td style="padding:8px 12px">Explica cada resultado y lo conecta con ideas ML</td><td style="padding:8px 12px">20%</td></tr></tbody></table>

<div style="background:#F8FAFC;border:1px solid #E2E8F0;border-radius:10px;padding:14px 18px;margin-top:14px"><div style="font-weight:800;color:#1E293B;margin-bottom:6px">08 · Referencias</div><ul style="margin:0 0 0 18px;font-size:13.5px;color:#334155;line-height:1.7"><li>Laplace, P.-S. (1814). <i>Essai philosophique sur les probabilités</i>. (Origen definición clásica.)</li><li>von Mises, R. (1928). <i>Probability, Statistics and Truth</i>. (Fundamento frecuentista.)</li><li>Wasserman, L. (2004). <i>All of Statistics</i>. Springer. (Ley grandes números; máxima verosimilitud.)</li><li>Downey, A. (2014). <i>Think Stats</i> (2.ª ed.). O’Reilly. (Estadística por simulación en Python.)</li><li>NumPy Developers. <i>NumPy Reference — Random Generator</i>. numpy.org/doc.</li></ul></div><div style="background:#4F46E5;color:white;border-radius:8px;padding:10px 16px;margin-top:12px;text-align:center;font-size:12px;letter-spacing:1px">INF672 · MACHINE LEARNING · UATF — GUÍA DE LABORATORIO 6</div><p style="font-size:13px;color:#64748B;margin-top:10px">Ejecutar celdas en orden con <code>Shift+Enter</code>. Exportar PDF: Jupyter → File → Print Preview → Save as PDF o <code>jupyter nbconvert --to pdf GL06_Coca_Abdair.ipynb</code>.</p>